# **AI Assisted Ballot Translator**

### Input:
- Voter's ballot (or address)

### Output:
- Each policy made to real

### Steps:
[Done] 1. Convert Missouri ballot titles to search keywords

[Done] 2. Enter search keywords onto Google

[Done] 3. Rank Google sources based on relevancy (cosine similarity to original ballot)

[Done] 4. Extract context from most relevant Google sources

[In Progress] 5. Q&A using context from most relevant Google sources

[Not Done] 6. Summarize answers and compile as final output

[Almost Done] 7. Connect to Gradio (UI)



# **Installs & Imports**

In [1]:
!pip install langchain langchain_community langchain-cohere -q
!pip install --upgrade serpapi -q
!pip install --upgrade gradio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.3/303.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
# Extract keywords from ballot title for search query libraries
import re
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
# Load a spaCy model for keyword extraction (could also use another NLP library)
nlp = spacy.load("en_core_web_sm")

# Web search libraries
import serpapi
from google.colab import userdata
SERPAPI_KEY = userdata.get("google_serpapi")
SERPAPI_CLIENT = serpapi.Client(api_key=SERPAPI_KEY)
import requests
from bs4 import BeautifulSoup

# Pretty print search results
import pprint

# Cosine similarity libraries
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
ENCODER = SentenceTransformer('all-MiniLM-L6-v2')

# LLM libraries
from transformers import pipeline
# Initialize Hugging Face summarization pipeline with T5 model
summarizer = pipeline("summarization", model="t5-small")
from langchain_cohere import ChatCohere
from langchain_core.messages import HumanMessage, SystemMessage

# RAG libraries
from langchain.text_splitter import CharacterTextSplitter
import json

# UI libraries
import gradio

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Device set to use cpu


# **Data**

In [3]:
# Inputs
ballot_measures_2014 = {
      "policies": [ """Shall the Missouri Constitution be amended so that the people shall be secure in their electronic communications and data from unreasonable searches and seizures as they are now likewise secure in their persons, homes, papers and effects? State and local governmental entities expect no significant costs or savings.""",
     """Shall the Missouri Constitution be amended to ensure that the right of Missouri citizens to engage in agricultural production and ranching practices shall not be infringed? The potential costs or savings to governmental entities are unknown, but likely limited unless the resolution leads to increased litigation costs and/or the loss of federal funding.""",
     """Shall the Missouri Constitution be amended to include a declaration that the right to keep and bear arms is a unalienable right and that the state government is obligated to uphold that right? State and local governmental entities should have no direct costs or savings from this proposal. However, the proposal’s passage will likely lead to increased litigation and criminal justice related costs. The total potential costs are unknown, but could be significant.""",
     """Should the Missouri Constitution be changed to enact a temporary sales tax of three-quarters of one percent to be used solely to fund state and local highways, roads, bridges and transportation projects for ten years, with priority given to repairing unsafe roads and bridges? This change is expected to produce $480 million annually to the state"s Transportation Safety and Job Creation Fund and $54 million for local governments. Increases in the gas tax will be prohibited. This revenue shall only be used for transportation purposes and cannot be diverted for other uses.""",
     """Shall the Missouri Constitution be amended to create a "Veterans Lottery Ticket" and to use the revenue from the sale of these tickets for projects and services related to veterans? The annual cost or savings to state and local governmental entities is unknown, but likely minimal. If sales of a veterans lottery ticket game decrease existing lottery ticket sales, the profits of which fund education, there could be a small annual shift in funding from education to veterans’ programs."""
       ]
}
ballot_measures_2024 = {
    "policies": ["""Do you want to amend the Missouri Constitution to: allow the Missouri Gaming Commission to regulate licensed sports wagering including online sports betting, gambling boats, professional sports betting districts and mobile licenses to sports betting operators; restrict sports betting to individuals physically located in the state and over the age of 21; allow license fees prescribed by the Commission and a 10% wagering tax on revenues received to be appropriated for education after expenses incurred by the Commission and required funding of the Compulsive Gambling Prevention Fund; and allow for the general assembly to enact laws consistent with this amendment? State governmental entities estimate onetime costs of $660,000, ongoing annual costs of at least $5.2 million, and initial license fee revenue of $11.75 million. Because the proposal allows for deductions against sports gaming revenues, they estimate unknown tax revenue ranging from $0 to $28.9 million annually. Local governments estimate unknown revenue.""",
                        """Do you want to amend the Missouri Constitution to: establish a right to make decisions about reproductive health care, including abortion and contraceptives, with any governmental interference of that right presumed invalid; remove Missoui’s ban on abortion; allow regulation of reproductive health care to improve or maintain the health of the patient; require the government not to discriminate, in government programs, funding, and other activities, against persons providing or obtaining reproductive health care; and allow abortion to be restricted or banned after Fetal Viability except to protect the life or health of the woman? State governmental entities estimate no costs or savings, but unknown impact. Local governmental entities estimate costs of at least $51,000 annually in reduced tax revenues. Opponents estimate a potentially significant loss to state revenue.""",
                        """Do you want to amend the Missouri Constitution to: allow the Missouri Gaming Commission to issue one additional gambling boat license to operate on the portion of the Osage River from the Missouri River to the Bagnell Dam; require the prescribed location shall include artificial spaces that contain water and are within 500 feet of the 100-year base flood elevation as established by the Federal Emergency Management Agency; and require all state revenues derived from the issuance of the gambling boat license shall be appropriated to early-childhood literacy programs in public institutions of elementary education? State governmental entities estimate one-time costs of $763,000, ongoing costs of $2.2 million annually, initial fee revenue of $271,000, ongoing admission and other fee revenue of $2.1 million annually, and annual gaming tax revenue of $14.3 million. Local governments estimate unknown revenue.""",
                        """Shall the Missouri Constitution be amended to preserve funding of law enforcement personnel for the administration of justice? State and local governmental entities estimate an unknown fiscal impact.""",
                        """Shall the Missouri Constitution be amended to: Make the Constitution consistent with state law by only allowing citizens of the United States to vote; Prohibit the ranking of candidates by limiting voters to a single vote per candidate or issue; and Require the plurality winner of a political party primary to be the single candidate at a general election? State and local governmental entities estimate no costs or savings.""",
                        """Do you want to amend Missouri law to: increase minimum wage January 1, 2025 to $13.75 per hour, increasing $1.25 per hour each year until 2026, when the minimum wage would be $15.00 per hour; adjust minimum wage based on changes in the Consumer Price Index each January beginning in 2027; require all employers to provide one hour of paid sick leave for every thirty hours worked; allow the Department of Labor and Industrial Relations to provide oversight and enforcement; and exempt governmental entities, political subdivisions, school districts and education institutions? State governmental entities estimate one-time costs ranging from $0 to $53,000, and ongoing costs ranging from $0 to at least $256,000 per year by 2027. State and local government tax revenue could change by an unknown annual amount depending on business decisions."""
    ]
}

In [4]:
from dataclasses import dataclass

@dataclass
class VotablePeople:
    people : list
    categories: dict
    office : list

@dataclass
class Ballot:
    policies : list #list of strings?
    # people : VotablePeople

    def from_dict(data): #save ballot/test ballot format...
        return Ballot(**data)
    def display(self):
        print( "The following policies are being voted on:")
        for pol in self.policies:
            print(pol)
            print("")

        # print( "The following people need to voted on:" )
        # self.people.display()


# **1. Convert to search keywords**

In [5]:
def extract_keywords_tfidf(text: str, n=5):
    """Extract top n keywords using TF-IDF"""
    vectorizer = TfidfVectorizer(stop_words="english", max_features=n)
    tfidf_matrix = vectorizer.fit_transform([text])
    feature_names = vectorizer.get_feature_names_out()
    return feature_names

def extract_keywords_spacy(text: str, n=5):
    """Extract top n keywords using spaCy (for better context)"""
    doc = nlp(text)
    keywords = [chunk.text for chunk in doc.noun_chunks]
    keywords = list(set(keywords))  # Remove duplicates
    keywords = keywords[:n]  # Return top N keywords
    keywords = list(set(keywords))  # Remove duplicates
    keywords = [keyword.lower() for keyword in keywords] # Convert keywords to all lowercase
    return keywords

def add_location_keywords(keywords, state: str, country: str):
    """Add state and country as keywords"""
    if state.lower() not in keywords:
        keywords.append(state.lower())
    if country.lower() not in keywords:
        keywords.append(country.lower())
    return keywords

def clean_text(text: str):
    """Clean the text (removes numbers & special characters)"""
    return re.sub(r"\d+|[^a-zA-Z\s]", "", re.sub(r"\s+", " ", text)).strip()

def generate_search_keywords(ballot_policy: str, state: str, country="USA", keyword_count=5):
    """Use TF-IDF and Spacy models for quick keyword extraction to form shorter search query"""
    tfidf_keywords = extract_keywords_tfidf(ballot_policy, keyword_count)
    search_query = " ".join(tfidf_keywords)
    nlp_keywords = extract_keywords_spacy(search_query, keyword_count)
    context_based_keywords = add_location_keywords(nlp_keywords, state, country)
    search_query = " ".join(context_based_keywords) # Join the keywords into the final search query
    search_query = clean_text(search_query)
    return search_query

# **2. Use keywords to search on Google**

In [6]:
def web_search(query: str):
    """Google web search query using SerpAPI"""
    search = SERPAPI_CLIENT.search({
        "engine": "google",
        "q": query,
    })
    return search

def summarize_web_search(query: str):
    """Run a web search on a single query and return a list of results."""
    search_result_summaries = []
    results = web_search(query).get("organic_results", [])
    for result in results:
        search_result_summaries.append({
            "Title": result.get("title"),
            "Snippet": result.get("snippet"),
            "Link": result.get("link")
        })

    return search_result_summaries

# **3. Rank Google sources based on relevancy (cosine similarity to original ballot)**

In [8]:
def text_cosine_similarity(text1, text2):
    # Get embeddings
    emb1 = ENCODER.encode([text1])
    emb2 = ENCODER.encode([text2])

    # Compute cosine similarity
    score = cosine_similarity(emb1, emb2)[0][0]
    return float(score)

def rank_similarity_scores(search_summaries, prompt):
    """Rank similarity scores for a single list of search summaries."""
    similarity_scores = []

    for summary in search_summaries:
        score = text_cosine_similarity(
            prompt,
            summary["Title"] + " " + summary["Snippet"]
        )
        similarity_scores.append((score, summary))

    ranked_scores = sorted(similarity_scores, key=lambda item: item[0], reverse=True)

    return ranked_scores

# **4. Extract context from most relevant Google sources**

In [10]:
def webscrape_url(url):
    """extract content from a webpage of url"""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        paragraphs = soup.find_all("p")
        content = " ".join(p.get_text() for p in paragraphs if p.get_text())
        return content if content else None
    except requests.exceptions.RequestException as e:
        print(f"Error fetching content from {url}: {e}")
        return None

def fetch_documents(ranked_scores):
    """Get cleaned documents from top cosine-similarity ranked search results"""
    ballot_documents = []
    for score, summary in ranked_scores:
        url = summary.get("Link")
        if not url:
            continue

        content = webscrape_url(url)
        if content:
            cleaned = clean_text(content)
            ballot_documents.append(cleaned)
    return ballot_documents  # returns empty list if nothing found

def chunk_document(document, chunk_size=1500, chunk_overlap=200):
    text_splitter = CharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = text_splitter.split_text(document)
    return chunks

def summarize_chunks(chunks, max_length=200, min_length=50):
    summaries = []
    for chunk in chunks:
        summary = summarizer(chunk, max_length=max_length, min_length=min_length, do_sample=False)
        summaries.append(summary[0]["summary_text"])
    final_summary = "\n".join(summaries)
    return final_summary

# **5. LLM Q&A using context from most relevant Google sources**


In [11]:
LLM_NAME = "command-r-plus-08-2024"
TEMPERATURE = 1 # or 0.5
COHERE_API_KEY = userdata.get("cohereAPI") # Name of my COHERE API key in google colab secrets. Change name in string to whatever you named your key
LLM = ChatCohere(model=LLM_NAME, max_tokens=300, temperature=TEMPERATURE, cohere_api_key=COHERE_API_KEY)

In [12]:
def LLM_summarize(ballot, final_summary):
    if not final_summary:
        return {"summary": "", "explanation": "No content available to explain."}

    prompt = f"""Task: Your goal is to rewrite the following ballot policy in clear, simple, and politically neutral language so that any voter can easily understand it.
Instructions:
1. Keep the explanation factual and politically neutral — do not add opinions or persuasive language.
2. Use plain English (aim for an 8th grade reading level - short sentences, everyday words, active voice).
3. If the ballot wording is unclear or uses vague legal phrasing, briefly note that (e.g., ‘The proposal does not specify how funds would be used’).
4. Maintain complete accuracy - only simplify the wording. Do not alter the meaning.
Here is ballot: {ballot}
Here is internet summary of context: {final_summary}
"""
    messages = [
        SystemMessage(content="""You're a politically neutral expert in civic communication and public policy. Explain the ballot measure so all voters can understand."""),
        HumanMessage(content=prompt)
    ]

    response = LLM.invoke(messages)
    return {"ballot": ballot, "LLM_summary": response.content, "internet_summary": final_summary}

def LLM_expand(ballot, final_summary):
    prompt = f"""Task: Your goal is to rewrite the following ballot policy in clear, simple, and politically neutral language so that any voter can easily understand it.
Instructions:
1. Keep the explanation factual and politically neutral — do not add opinions or persuasive language.
2. Use plain English (aim for an 8th grade reading level - short sentences, everyday words, active voice).
3. If the ballot wording is unclear or uses vague legal phrasing, briefly note that (e.g., ‘The proposal does not specify how funds would be used’).
4. Include factual context about the ballot:
    - Who is promoting the ballot (individuals, organizations, political parties)?
    - Who supports it (names, organizations, political parties)?
    - Who opposes it (names, organizations, political parties)?
    - Any reported funding sources for promotion or opposition and amounts, if publicly available.
5. Maintain complete accuracy - only simplify the wording. Do not alter the meaning.
6. Keep the summary concise and easy to read, focusing on clear facts.

Here is the ballot: {ballot}
Here is internet summary of context: {final_summary}
"""
    messages = [
        SystemMessage(content="""You're a politically neutral expert in civic communication and public policy. Explain the ballot measure so all voters can understand."""),
        HumanMessage(content=prompt)
    ]

    expanded_response = LLM.invoke(messages)
    return expanded_response.content

# **6. Combine steps 1-5 into 1 function**

In [14]:
def explain_ballot(ballot, state, country, search_keyword_count=5):
    # 1. Generate search query of search_keyword_count number of keywords
    search_query = generate_search_keywords(ballot, state, country, search_keyword_count)

    # 2. Run google web search
    search_results = summarize_web_search(search_query)

    # 3. Rank similarity scores of search results' descriptions against actual ballot text
    ranked_scores = rank_similarity_scores(search_results, ballot)

    # 4. Extract text from websites & summarize all of them
    ballot_documents = fetch_documents(ranked_scores)
    # If no docs found → skip summarizing
    if not ballot_documents:
        final_summary = "!Couldn't webscrape websites to generate summary!"
    else:
        summaries = []
        for document in ballot_documents:
            chunks = chunk_document(document)
            summary = summarize_chunks(chunks)
            summaries.append(summary)
        final_summary = "\n* ".join(summaries) # Combine summaries from all searched documents

    # 5. LLM intelligently explain ballot
    response = LLM_summarize(ballot, final_summary) # LLM Summarize
    expanded_response = LLM_expand(ballot, final_summary) # LLM Extract interested groups & funding
    response["LLM_expanded"] = expanded_response
    return response

# **7. Connect to Gradio (UI)**

In [15]:
%reload_ext gradio

In [20]:
def explain_ballot(ballot, state, country, search_keyword_count=5):
    # 1. Generate search query of search_keyword_count number of keywords
    search_query = generate_search_keywords(ballot, state, country, search_keyword_count)

    # 2. Run google web search
    search_results = summarize_web_search(search_query)

    # 3. Rank similarity scores of search results' descriptions against actual ballot text
    ranked_scores = rank_similarity_scores(search_results, ballot)

    # 4. Extract text from websites & summarize all of them
    ballot_documents = fetch_documents(ranked_scores)

    if not ballot_documents:
        final_summary = "!Couldn't webscrape websites to generate summary!"
    else:
        summaries = []
        for document in ballot_documents:
            chunks = chunk_document(document)
            summary = summarize_chunks(chunks)
            summaries.append(summary)
        final_summary = "\n* ".join(summaries)

    # 5. LLM intelligently explain ballot
    response = LLM_summarize(ballot, final_summary)
    expanded_response = LLM_expand(ballot, final_summary)
    response["LLM_expanded"] = expanded_response

    return response


def explain(ballot_input, state_input="Missouri"):
    response = explain_ballot(ballot_input, state_input, "USA", search_keyword_count=5)

    summary = response.get("LLM_summary", "")
    expansion = response.get("LLM_expanded", "")

    return summary, expansion, response


# List of all 50 US states
us_states = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut",
    "Delaware", "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa",
    "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan",
    "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire",
    "New Jersey", "New Mexico", "New York", "North Carolina", "North Dakota", "Ohio", "Oklahoma",
    "Oregon", "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota", "Tennessee",
    "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"
]

# Create Gradio Blocks interface
with gradio.Blocks() as demo:

    state_input = gradio.Dropdown(
        choices=us_states,
        label="Select a U.S. State",
        value="Missouri",
        interactive=True
    )

    ballot_input = gradio.Textbox(
        label="Ballot measure/title",
        lines=4,
        placeholder="What will be seen at the voting booth",
        interactive=True
    )

    explain_btn = gradio.Button("Learn more")

    with gradio.Tab("LLM Summary"):
        summary = gradio.Markdown()

    with gradio.Tab("LLM elaboration"):
        elaboration = gradio.Markdown()

    with gradio.Tab("Full JSON response"):
        full_output_json = gradio.JSON()

    explain_btn.click(
        fn=explain,
        inputs=[ballot_input, state_input],
        outputs=[summary, elaboration, full_output_json],
        api_name="explain"
    )

    examples = gradio.Examples(
        examples=[[policy, "Missouri"] for policy in ballot_measures_2024["policies"]],
        inputs=[ballot_input, state_input],
        outputs=[summary, elaboration, full_output_json],
        fn=explain
    )

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://29beafdb3a539930cf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Error fetching content from https://missouriindependent.com/ballot-measures/amendment-7/: 403 Client Error: Forbidden for url: https://missouriindependent.com/ballot-measures/amendment-7/


Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://29beafdb3a539930cf.gradio.live


# Run all ballots (no UI)

In [ ]:
# Final code all added together (without UI)
ballots = ballot_measures_2014["policies"] + ballot_measures_2024["policies"]
state = "missouri"
country = "USA"
search_keyword_count = 5
full_responses = []
for i, ballot in enumerate(ballots):

    # 1. Generate search query of search_keyword_count number of keywords
    search_query = generate_search_keywords(ballot, state, country, search_keyword_count)

    # 2. Run google web search
    search_results = summarize_web_search(search_query)

    # 3. Rank similarity scores of search results' descriptions against actual ballot text
    ranked_scores = rank_similarity_scores(search_results, ballot)

    # 4. Extract text from websites & summarize all of them
    ballot_documents = fetch_documents(ranked_scores)
    # If no docs found → skip summarizing
    if not ballot_documents:
        final_summary = "!Couldn't webscrape websites to generate summary!"
        continue
    else:
        summaries = []
        for document in ballot_documents:
            chunks = chunk_document(document)
            summary = summarize_chunks(chunks)
            summaries.append(summary)
        final_summary = "\n* ".join(summaries) # Combine summaries from all searched documents

    # 5. LLM intelligently explain & elaborate on ballot
    response = LLM_summarize(ballot, final_summary) # LLM Summarize
    expanded_response = LLM_expand(ballot, final_summary) # LLM Extract interested groups & funding
    response["LLM_expanded"] = expanded_response
    full_responses.append(response)

    print(f"Ballot measure {i}: {ballot}")
    print(f"Search keywords (query): {search_query}")
    pprint.pprint(ranked_scores)
    print(f"\n\nFINAL SUMMARY (from internet only, no LLM): {final_summary}")
    print(f"\nFINAL SUMMARY (with LLM): {response["LLM_summary"]}")
    print(f"\nEXTRA SUMMARY (with LLM): {response["LLM_expanded"]}")
    print("-" * 50)
    print()

Error fetching content from https://www.workplaceprivacyreport.com/2014/08/articles/workplace-investigations/missouri-constitutional-amendment-protects-electronic-privacy/: 403 Client Error: Forbidden for url: https://www.workplaceprivacyreport.com/2014/08/articles/workplace-investigations/missouri-constitutional-amendment-protects-electronic-privacy/
Error fetching content from https://time.com/3087608/missouri-electronic-privacy-amendment/: 406 Client Error: Not Acceptable for url: https://time.com/3087608/missouri-electronic-privacy-amendment/
Error fetching content from https://www.dwt.com/gcp/states/missouri: 403 Client Error: Forbidden for url: https://www.dwt.com/gcp/states/missouri
Error fetching content from https://revisor.mo.gov/main/OneSection.aspx?section=351.609: HTTPSConnectionPool(host='revisor.mo.gov', port=443): Max retries exceeded with url: /main/OneSection.aspx?section=351.609 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7ce19b0187

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Ballot measure 0: Shall the Missouri Constitution be amended so that the people shall be secure in their electronic communications and data from unreasonable searches and seizures as they are now likewise secure in their persons, homes, papers and effects? State and local governmental entities expect no significant costs or savings.
Search keywords (query): amended communications data missouri usa
[(0.8057214021682739,
  {'Link': 'https://www.workplaceprivacyreport.com/2014/08/articles/workplace-investigations/missouri-constitutional-amendment-protects-electronic-privacy/',
   'Snippet': 'That the people shall be secure in their persons, papers, '
              'homes, effects, and electronic communications and data, from '
              'unreasonable searches and ...',
   'Title': 'Missouri Constitutional Amendment Protects Electronic Privacy'}),
 (0.7657653093338013,
  {'Link': 'https://ballotpedia.org/Missouri_Electronic_Data_Protection,_Amendment_9_(August_2014)',
   'Snippet': 'Th

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Ballot measure 1: Shall the Missouri Constitution be amended to ensure that the right of Missouri citizens to engage in agricultural production and ranching practices shall not be infringed? The potential costs or savings to governmental entities are unknown, but likely limited unless the resolution leads to increased litigation costs and/or the loss of federal funding.
Search keywords (query): agricultural constitution costs missouri usa
[(0.7841488718986511,
  {'Link': 'https://mofarmerscare.com/farming-rights-amendment/',
   'Snippet': 'The amendment aims to guarantee the right to farm and ranch, '
              'protecting farm families, jobs, and ensuring access to food, '
              'while protecting from out-of-state ...',
   'Title': 'Missouri Farming Rights Amendment'}),
 (0.7803464531898499,
  {'Link': 'https://ballotpedia.org/Missouri_Amendment_1,_Constitutional_Right_to_Farm_Measure_(August_2014)',
   'Snippet': 'The measure explicitly guarantees farmers and ranchers the

...

Ballot measure 8: Shall the Missouri Constitution be amended to preserve funding of law enforcement personnel for the administration of justice? State and local governmental entities estimate an unknown fiscal impact.
Search keywords (query): constitution enforcement entities missouri usa
[(0.6008269786834717,
  {'Link': 'https://www.supremecourt.gov/DocketPDF/24/24-796/348680/20250226100808914_24-796%20Amici%20Brief%20of%20Montana%20et%20al.pdf',
   'Snippet': 'Under Missouri law, state officials cannot use state resources '
              'to enforce certain federal laws. In response to a suit filed by '
              'the Federal ...',
   'Title': 'Missouri v. US'}),
 (0.5881726145744324,
  {'Link': 'https://ago.mo.gov/divisions/governmental-affairs/',
   'Snippet': "The Governmental Affairs Section's mission and objectives are "
              "to: defend Missouri's Constitution, statutes, and state "
              'agencies, through representing the State ...',
   'Title': 'Governme

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Ballot measure 9: Shall the Missouri Constitution be amended to: Make the Constitution consistent with state law by only allowing citizens of the United States to vote; Prohibit the ranking of candidates by limiting voters to a single vote per candidate or issue; and Require the plurality winner of a political party primary to be the single candidate at a general election? State and local governmental entities estimate no costs or savings.
Search keywords (query): candidate constitution single state vote missouri usa
[(0.7029396891593933,
  {'Link': 'https://www.youtube.com/watch?v=WL_U8CcCEGU',
   'Snippet': 'Amendment 7 would prohibit non-citizens from voting and ban '
              'ranked-choice voting. This ban would not apply to any cities '
              'that already have ...',
   'Title': "What to know about Amendment 7 on Missouri's ballot"}),
 (0.6974742412567139,
  {'Link': 'https://ballotpedia.org/Missouri_Single-Vote_and_Citizenship_Voting_Requirements,_One_Candidate_Per_

Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Ballot measure 10: Do you want to amend Missouri law to: increase minimum wage January 1, 2025 to $13.75 per hour, increasing $1.25 per hour each year until 2026, when the minimum wage would be $15.00 per hour; adjust minimum wage based on changes in the Consumer Price Index each January beginning in 2027; require all employers to provide one hour of paid sick leave for every thirty hours worked; allow the Department of Labor and Industrial Relations to provide oversight and enforcement; and exempt governmental entities, political subdivisions, school districts and education institutions? State governmental entities estimate one-time costs ranging from $0 to $53,000, and ongoing costs ranging from $0 to at least $256,000 per year by 2027. State and local government tax revenue could change by an unknown annual amount depending on business decisions.
Search keywords (query): governmental hour minimum wage missouri usa
[(0.8407207727432251,
  {'Link': 'https://www.workstream.us/wage-inde